# Matplotlib fundamentals

Almost every Python chart eventually becomes a Matplotlib figure, even when a higher-level library
draws it. Learning the anatomy of `Figure` and `Axes` — and the four workhorse chart types — pays
off everywhere. This notebook uses the Gapminder panel.

## Learning objectives

By the end of this notebook you will be able to:

- create a figure and axes explicitly and label every axis;
- draw line, area, histogram, and bar charts;
- aggregate data before plotting so the chart shows one line per category;
- control colour, titles, and layout;
- explain when each chart type is the right choice.

## Concept

Matplotlib separates the **Figure** (the whole canvas) from the **Axes** (one plot area with its
own x and y axes). Most confusion comes from mixing the two. `fig, ax = plt.subplots()` gives you
both, and from then on you call methods on `ax`. Good charts always have a title, axis labels with
units, and a readable colour choice.

Chart choice follows the question:

- a **line** shows a trend over an ordered variable, usually time;
- an **area** chart is a line with the space below filled, good for totals over time;
- a **histogram** shows the distribution of one numeric variable;
- a **bar** chart compares a numeric value across discrete categories.

Plotting raw rows is usually wrong: aggregate to the level the question asks about first. For
"how has life expectancy changed?", the answer is a mean per year, not one line per country.

## Worked example

### Load and prepare

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from ds_practice import load_gapminder, set_seed, set_theme

set_seed(42)
set_theme()
gap = load_gapminder()
print("shape:", gap.shape)
display(gap.head())

### A line chart: life expectancy over time

We aggregate to the world mean per year, then plot it.

In [ ]:
world = gap.groupby("year")["lifeExp"].mean().reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(world["year"], world["lifeExp"], marker="o", color="#4c72b0")
ax.set_title("World mean life expectancy, 1952–2007")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
fig.tight_layout()
plt.show()

### Several lines by continent

One line per continent shows the gap between regions that a world mean hides.

In [ ]:
by_continent = gap.groupby(["year", "continent"])["lifeExp"].mean().reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
for name, group in by_continent.groupby("continent"):
    ax.plot(group["year"], group["lifeExp"], marker="o", label=name)
ax.set_title("Life expectancy by continent")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
ax.legend(title="Continent", frameon=False)
fig.tight_layout()
plt.show()

### An area chart

An area chart fills the space under a line. It suits a cumulative total; here we show population
growth for one continent.

In [ ]:
asia = gap[gap["continent"] == "Asia"].groupby("year")["pop"].sum().reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(asia["year"], asia["pop"] / 1e9, color="#dd8452", alpha=0.6)
ax.plot(asia["year"], asia["pop"] / 1e9, color="#dd8452")
ax.set_title("Asia population")
ax.set_xlabel("Year")
ax.set_ylabel("Population (billions)")
fig.tight_layout()
plt.show()

### A histogram

The distribution of life expectancy in the latest year is left-skewed: most countries are high,
with a tail of lower values.

In [ ]:
latest = gap[gap["year"] == gap["year"].max()]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(latest["lifeExp"], bins=20, color="#55a868", edgecolor="white")
ax.set_title(f"Life expectancy in {int(latest['year'].iloc[0])}")
ax.set_xlabel("Life expectancy (years)")
ax.set_ylabel("Number of countries")
fig.tight_layout()
plt.show()

### A bar chart

Bars compare categories. We show the mean life expectancy per continent, sorted for readability.

In [ ]:
means = latest.groupby("continent")["lifeExp"].mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(means.index, means.values, color="#c44e52")
ax.set_title("Mean life expectancy by continent")
ax.set_xlabel("Life expectancy (years)")
fig.tight_layout()
plt.show()

## Exercises

1. **GDP trend.** Plot world mean GDP per capita over time on a log y-axis and explain why a log
   scale is appropriate for a quantity that grows multiplicatively.
2. **Distribution shift.** Overlay histograms of life expectancy for 1952 and 2007 with
   `alpha=0.5`. Describe how the distribution has moved.
3. **Small multiples.** Use a 2×3 grid of axes (`fig, axes = plt.subplots(2, 3)`) to draw one
   continent's life-expectancy trend per panel. Label each panel.

## Limitations

Matplotlib's defaults are functional, not beautiful, and producing a publication figure takes
deliberate styling. The Gapminder panel is five-yearly, so lines are interpolations rather than
observed annual values. Bar charts hide distribution: two categories with the same mean can differ
enormously in spread, which a box plot would reveal. Colour palettes must be checked for
accessibility, and a chart without axis units is misleading.